# GRPO Training (Spelling Bee and Wordle)

In [1]:
import os, sys

REPO_URL = "https://github.com/jackiepiepkorn/cse291-nytgames.git"
REPO_DIR = "/kaggle/working/cse291-nytgames"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

Cloning into '/kaggle/working/cse291-nytgames'...
remote: Enumerating objects: 178, done.
remote: Counting objects: 100% (178/178), done.
remote: Compressing objects: 100% (111/111), done.
remote: Total 178 (delta 80), reused 153 (delta 58), pack-reused 0 (from 0)
Receiving objects: 100% (178/178), 1.52 MiB | 5.98 MiB/s, done.
Resolving deltas: 100% (80/80), done.


In [ ]:
%pip install torch transformers datasets accelerate trl>=0.15 peft>=0.13 bitsandbytes gymnasium pandas wordfreq --quiet

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import re
import json
import math
import random
import torch
import numpy as np
from pathlib import Path
from collections import Counter
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel
from trl import GRPOConfig, GRPOTrainer

from nytgames import (
    SpellingBeeConfig, SpellingBeeEnv, SpellingBeeDataset,
    WordleConfig, WordleEnv, load_dictionary,
)
from nytgames.env.wordle import _score_guess, _format_feedback
from nytgames.data.dataset import _SPELLING_BEE_CSV_PATH as SB_CSV_PATH, _PROMPTS_DIR

# Prompts are class attributes on SpellingBeeDataset, not module-level vars
SB_SYSTEM_PROMPT = SpellingBeeDataset._SYSTEM_PROMPT
SB_USER_PROMPT_TEMPLATE = SpellingBeeDataset._USER_PROMPT_TEMPLATE

WORDLE_SYSTEM_PROMPT = (_PROMPTS_DIR / "wordle_system.md").read_text().strip()
WORDLE_USER_PROMPT_TEMPLATE = (_PROMPTS_DIR / "wordle_user.md").read_text().strip()

_DATA_DIR = Path(SB_CSV_PATH).parent
WORDLE_SOLUTIONS_PATH = _DATA_DIR / "wordle_past_solutions.txt"

# --- [Improvement 3] Word frequency prior ---
try:
    from wordfreq import zipf_frequency
    HAS_WORDFREQ = True
    print("wordfreq library available — frequency prior enabled")
except ImportError:
    HAS_WORDFREQ = False
    print("wordfreq not installed — frequency prior disabled (pip install wordfreq)")

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla P100-PCIE-16GB
VRAM: 17.1 GB


## 2. Configuration

In [ ]:
GAME = "spelling_bee"

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR = "./grpo_output"

NUM_GENERATIONS = 8
MAX_COMPLETION_LENGTH = 16   # spelling bee words are 4-15 letters; saves generation time
TEMPERATURE = 1.0            # slightly lower than 1.2 for better quality exploration
BETA = 0.04

LEARNING_RATE = 5e-6
NUM_EPOCHS = 3               # multiple passes over fewer puzzles for stronger signal
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 4
USE_BF16 = torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False

USE_LORA = True
LORA_R = 16
LORA_ALPHA = 32

MAX_PUZZLES = 400            # more puzzles for better GRPO diversity (was 200)
CSV_PATH = SB_CSV_PATH

print(f"Game: {GAME}")
print(f"Model: {MODEL_NAME}")
print(f"BF16: {USE_BF16}, LoRA: {USE_LORA} (r={LORA_R})")
print(f"Group size G={NUM_GENERATIONS}, β={BETA}, lr={LEARNING_RATE}")
print(f"Puzzles: {MAX_PUZZLES or 'all'}, Epochs: {NUM_EPOCHS}, Max tokens: {MAX_COMPLETION_LENGTH}")

Game: spelling_bee
Model: Qwen/Qwen2.5-1.5B-Instruct
BF16: True, LoRA: True (r=16)
Group size G=8, β=0.04, lr=5e-06
Puzzles: 200, Epochs: 3, Max tokens: 16


## 2.1 Dictionary Caching & Candidate Precomputation (Improvements 1, 9)

In [ ]:
# ============================================================================
# [Improvement 9] Cache dictionary features — precompute bitmasks, letter
# frequencies, and entropy scores so legality checks during training are fast.
# [Improvement 1] Restrict action space — precompute valid candidate words
# per puzzle so the model selects from dictionary instead of inventing words.
# ============================================================================

FULL_DICTIONARY = load_dictionary()

# --- Bitmask cache: each word → 26-bit int where bit i = letter (A+i) present ---
def _letter_bitmask(word: str) -> int:
    mask = 0
    for ch in word.upper():
        if 'A' <= ch <= 'Z':
            mask |= (1 << (ord(ch) - ord('A')))
    return mask

WORD_BITMASK_CACHE = {w: _letter_bitmask(w) for w in FULL_DICTIONARY}

# --- Letter frequency cache ---
_ALL_TEXT = "".join(FULL_DICTIONARY)
LETTER_FREQ = {ch: _ALL_TEXT.count(ch) / len(_ALL_TEXT) for ch in "ABCDEFGHIJKLMNOPQRSTUVWXYZ"}

# --- Entropy score per word (how much letter diversity it has) ---
def _word_entropy(word: str) -> float:
    counts = Counter(word.upper())
    total = len(word)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())

WORD_ENTROPY_CACHE = {w: _word_entropy(w) for w in FULL_DICTIONARY}

# --- [Improvement 1] Pre-filter Spelling Bee candidates per puzzle ---
def get_sb_candidates(allowed_letters: set, center_letter: str, min_len: int = 4) -> list:
    """Return all dictionary words that are valid for a given Spelling Bee puzzle."""
    allowed_upper = {l.upper() for l in allowed_letters}
    center_upper = center_letter.upper()
    allowed_mask = 0
    for l in allowed_upper:
        allowed_mask |= (1 << (ord(l) - ord('A')))
    center_bit = 1 << (ord(center_upper) - ord('A'))

    candidates = []
    for word, wmask in WORD_BITMASK_CACHE.items():
        if len(word) >= min_len and (wmask & ~allowed_mask) == 0 and (wmask & center_bit):
            candidates.append(word)
    return candidates

# --- [Improvement 1] Pre-filter Wordle valid guesses ---
WORDLE_VALID_GUESSES_SET = load_dictionary(length=5)

# --- [Improvement 3] Word frequency helper ---
def word_frequency_score(word: str) -> float:
    """Return a frequency-based score for the word (higher = more common)."""
    if HAS_WORDFREQ:
        return zipf_frequency(word.lower(), "en")
    return 0.0

# --- Wordle entropy / information gain helpers ---
def wordle_filter_candidates(candidates: set, guess: str, target: str) -> set:
    """Given a guess and target, return the candidates consistent with the feedback."""
    tiles = _score_guess(guess, target)
    return {c for c in candidates if _score_guess(guess, c) == tiles}

def wordle_info_gain(guess: str, candidates: set) -> float:
    """Compute information gain of a guess: log2(|before|) - E[log2(|after|)]."""
    if len(candidates) <= 1:
        return 0.0
    # Group candidates by the feedback pattern they'd produce
    pattern_groups = {}
    for c in candidates:
        pattern = tuple(_score_guess(guess, c))
        pattern_groups.setdefault(pattern, []).append(c)

    expected_log = 0.0
    n = len(candidates)
    for group in pattern_groups.values():
        p = len(group) / n
        expected_log += p * math.log2(max(len(group), 1))

    return math.log2(n) - expected_log

print(f"Dictionary cached: {len(FULL_DICTIONARY)} words")
print(f"Bitmask cache: {len(WORD_BITMASK_CACHE)} entries")
print(f"5-letter words for Wordle: {len(WORDLE_VALID_GUESSES_SET)}")

# Quick test: SB candidates for a sample puzzle
test_candidates = get_sb_candidates({"W", "A", "H", "O", "R", "T", "Y"}, "W")
print(f"Sample SB candidates (center=W, letters=WAHORTY): {len(test_candidates)} words")
print(f"  Examples: {test_candidates[:10]}")

## 3. Reward Functions

In [ ]:
# ============================================================================
# REWARD FUNCTIONS
# [Improvement 2] Dense/partial rewards instead of all-or-nothing
# [Improvement 3] Word frequency prior
# [Improvement 4] Wordle information gain reward
# [Improvement 7] Penalize repeated guesses
# ============================================================================

def _extract_text(completion) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        for msg in reversed(completion):
            if isinstance(msg, dict) and msg.get("content"):
                return msg["content"]
        return ""
    return str(completion)


# --- [Improvement 11/12] Extract answer from thinking format ---
def _extract_answer(text: str) -> str:
    """If the model uses 'Answer: <word>' format, extract just the answer.
    Otherwise return the whole text stripped."""
    # Try to extract from "Answer: <word>" format
    m = re.search(r"(?:Answer|ANSWER|answer)\s*:\s*([A-Za-z]+)", text)
    if m:
        return m.group(1).strip()
    # Otherwise take the last word-like token
    clean = text.strip()
    # If multi-line, take last line
    lines = [l.strip() for l in clean.split('\n') if l.strip()]
    if lines:
        last = lines[-1]
        words = re.findall(r'[A-Za-z]+', last)
        if words:
            return words[-1]
    return clean


# ===================== SPELLING BEE REWARDS =====================

def spelling_bee_format_reward(completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        clean = _extract_answer(_extract_text(completion))
        if re.fullmatch(r"[A-Za-z]+", clean):
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


def spelling_bee_letter_reward(completions, **kwargs) -> list[float]:
    """[Improvement 2] Partial rewards for letter validity.
    +0.3 if all letters are in the allowed set
    +0.3 if the center letter is present
    +0.1 if length >= 4
    +0.3 bonus if ALL three criteria met"""
    letters_list = kwargs.get("letters", [])
    centers = kwargs.get("center", [])
    rewards = []
    for i, completion in enumerate(completions):
        word = _extract_answer(_extract_text(completion)).upper()
        allowed = set(letters_list[i].upper()) if isinstance(letters_list[i], str) else set(l.upper() for l in letters_list[i])
        center = centers[i].upper()

        if not re.fullmatch(r"[A-Za-z]+", word):
            rewards.append(0.0)
            continue

        reward = 0.0
        uses_only_allowed = set(word) <= allowed
        has_center = center in word
        long_enough = len(word) >= 4

        if uses_only_allowed:
            reward += 0.3
        if has_center:
            reward += 0.3
        if long_enough:
            reward += 0.1
        if uses_only_allowed and has_center and long_enough:
            reward += 0.3  # bonus for meeting all criteria

        rewards.append(reward)
    return rewards


def spelling_bee_dictionary_reward(completions, **kwargs) -> list[float]:
    solutions_list = kwargs.get("valid_words", [])
    rewards = []
    for i, completion in enumerate(completions):
        word = _extract_answer(_extract_text(completion)).lower()
        solutions = set(s.strip().lower() for s in solutions_list[i].replace(";", ",").split(","))
        if word in solutions:
            rewards.append(3.0)  # strong signal for valid puzzle words
        else:
            rewards.append(0.0)
    return rewards


def spelling_bee_length_reward(completions, **kwargs) -> list[float]:
    solutions_list = kwargs.get("valid_words", [])
    rewards = []
    for i, completion in enumerate(completions):
        word = _extract_answer(_extract_text(completion)).lower()
        solutions = set(s.strip().lower() for s in solutions_list[i].replace(";", ",").split(","))
        if word not in solutions:
            rewards.append(0.0)
            continue
        if len(word) == 4:
            score = 1.0
        else:
            score = float(len(word))
        rewards.append(score / 10.0)
    return rewards


def spelling_bee_pangram_reward(completions, **kwargs) -> list[float]:
    """[Improvement 2] Pangram reward: 2.0 instead of 1.0 for stronger signal."""
    letters_list = kwargs.get("letters", [])
    solutions_list = kwargs.get("valid_words", [])
    rewards = []
    for i, completion in enumerate(completions):
        word = _extract_answer(_extract_text(completion)).lower()
        solutions = set(s.strip().lower() for s in solutions_list[i].replace(";", ",").split(","))
        if word not in solutions:
            rewards.append(0.0)
            continue
        allowed = set(letters_list[i].lower()) if isinstance(letters_list[i], str) else set(l.lower() for l in letters_list[i])
        if set(word) >= allowed:
            rewards.append(2.0)  # [Improvement 2] stronger pangram reward
        else:
            rewards.append(0.0)
    return rewards


def spelling_bee_frequency_reward(completions, **kwargs) -> list[float]:
    """[Improvement 3] Word frequency prior — reward common English words more.
    Uses zipf_frequency from wordfreq if available."""
    solutions_list = kwargs.get("valid_words", [])
    rewards = []
    for i, completion in enumerate(completions):
        word = _extract_answer(_extract_text(completion)).lower()
        solutions = set(s.strip().lower() for s in solutions_list[i].replace(";", ",").split(","))
        if word not in solutions:
            rewards.append(0.0)
            continue
        freq_score = word_frequency_score(word)
        rewards.append(0.1 * freq_score)
    return rewards


# ===================== WORDLE REWARDS =====================

def wordle_format_reward(completions, **kwargs) -> list[float]:
    rewards = []
    for completion in completions:
        word = _extract_answer(_extract_text(completion))
        if re.fullmatch(r"[A-Za-z]{5}", word):
            rewards.append(1.0)
        else:
            rewards.append(0.0)
    return rewards


def wordle_letter_match_reward(completions, **kwargs) -> list[float]:
    """[Improvement 2] Partial rewards: green=2pts, yellow=1pt, normalized."""
    targets = kwargs.get("target_word", [])
    rewards = []
    for i, completion in enumerate(completions):
        guess = _extract_answer(_extract_text(completion)).upper()
        target = targets[i].upper()
        if len(guess) != 5:
            rewards.append(0.0)
            continue
        green = sum(1 for g, t in zip(guess, target) if g == t)
        remaining_target = [t for g, t in zip(guess, target) if g != t]
        yellow = 0
        for g, t in zip(guess, target):
            if g != t and g in remaining_target:
                yellow += 1
                remaining_target.remove(g)
        score = (green * 2.0 + yellow * 1.0) / 10.0
        rewards.append(score)
    return rewards


def wordle_exact_match_reward(completions, **kwargs) -> list[float]:
    targets = kwargs.get("target_word", [])
    rewards = []
    for i, completion in enumerate(completions):
        guess = _extract_answer(_extract_text(completion)).upper()
        target = targets[i].upper()
        rewards.append(1.0 if guess == target else 0.0)
    return rewards


def wordle_info_gain_reward(completions, **kwargs) -> list[float]:
    """[Improvement 4] Information gain reward for Wordle.
    Measures how much a guess narrows down the candidate set.
    Uses the full 5-letter dictionary as candidate set for first guesses."""
    targets = kwargs.get("target_word", [])
    rewards = []
    for i, completion in enumerate(completions):
        guess = _extract_answer(_extract_text(completion)).upper()
        target = targets[i].upper()
        if len(guess) != 5 or guess not in WORDLE_VALID_GUESSES_SET:
            rewards.append(0.0)
            continue
        # Use a sample of candidates for efficiency (full set is too slow)
        sample_size = min(500, len(WORDLE_VALID_GUESSES_SET))
        candidate_sample = set(random.sample(list(WORDLE_VALID_GUESSES_SET), sample_size))
        if target not in candidate_sample:
            candidate_sample.add(target)
        ig = wordle_info_gain(guess, candidate_sample)
        # Normalize: max info gain is ~log2(500) ≈ 9
        rewards.append(ig / 9.0)
    return rewards


def wordle_repeat_penalty_reward(completions, **kwargs) -> list[float]:
    """[Improvement 7] Penalize repeated guesses within a batch.
    If the model generates duplicates within the same group, penalize them."""
    rewards = []
    seen = set()
    for completion in completions:
        guess = _extract_answer(_extract_text(completion)).upper()
        if guess in seen:
            rewards.append(-1.0)
        else:
            rewards.append(0.0)
        seen.add(guess)
    return rewards


# ---------- Shared reward: is it a real English word? ----------
VALID_ENGLISH_WORDS = load_dictionary()

def real_word_reward(completions, **kwargs) -> list[float]:
    """Reward 0.5 if the completion is any real English word (≥4 letters)."""
    rewards = []
    for completion in completions:
        word = _extract_answer(_extract_text(completion)).upper()
        if len(word) >= 4 and word in VALID_ENGLISH_WORDS:
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards


# ---------- Wordle: valid guess from word list ----------
WORDLE_VALID_GUESSES = load_dictionary(length=5)

def wordle_valid_guess_reward(completions, **kwargs) -> list[float]:
    """Reward 0.5 if the guess is a valid 5-letter word in the Wordle dictionary."""
    rewards = []
    for completion in completions:
        word = _extract_answer(_extract_text(completion)).upper()
        if word in WORDLE_VALID_GUESSES:
            rewards.append(0.5)
        else:
            rewards.append(0.0)
    return rewards


# ---------- [Improvement 12] Candidate ranking reward ----------
def spelling_bee_candidate_rank_reward(completions, **kwargs) -> list[float]:
    """Reward based on how the word ranks among candidates for this puzzle.
    Words in the puzzle solution set rank highest; real dictionary words that
    fit the letter constraints rank medium; everything else ranks 0."""
    letters_list = kwargs.get("letters", [])
    centers = kwargs.get("center", [])
    solutions_list = kwargs.get("valid_words", [])
    rewards = []
    for i, completion in enumerate(completions):
        word = _extract_answer(_extract_text(completion)).upper()
        solutions = set(s.strip().upper() for s in solutions_list[i].replace(";", ",").split(","))
        allowed = set(letters_list[i].upper()) if isinstance(letters_list[i], str) else set(l.upper() for l in letters_list[i])
        center = centers[i].upper()

        if word in solutions:
            # Top rank: actual puzzle solution
            base = 1.0
            # Bonus for longer words and pangrams
            if set(word) >= allowed:
                base += 0.5  # pangram bonus
            base += min(len(word) / 15.0, 0.5)  # length bonus
            rewards.append(base)
        elif (len(word) >= 4 and set(word) <= allowed and center in word
              and word in VALID_ENGLISH_WORDS):
            # Medium rank: valid dictionary word that fits constraints
            rewards.append(0.3)
        elif len(word) >= 4 and set(word) <= allowed and center in word:
            # Low rank: fits constraints but not a real word
            rewards.append(0.1)
        else:
            rewards.append(0.0)
    return rewards

### 3.1 Test Reward Functions

In [6]:
test_completions = [
    [{"role": "assistant", "content": "throw"}],
    [{"role": "assistant", "content": "ARROW"}],
    [{"role": "assistant", "content": "xyz!!"}],
    [{"role": "assistant", "content": "thaw"}],
    [{"role": "assistant", "content": "thataway"}],
]
test_kwargs = {
    "letters": ["WAHORTY"] * 5,
    "center": ["W"] * 5,
    "valid_words": ["arrow;arrowroot;athwart;away;awry;harrow;thataway;thaw;throw;throwaway;thwart;wahoo;wart;warty;wary;watt;what;whoa;woot;worry;worrywart;wort;worth;worthy;wrath;yarrow"] * 5,
}

print("Format rewards: ", spelling_bee_format_reward(test_completions, **test_kwargs))
print("Letter rewards: ", spelling_bee_letter_reward(test_completions, **test_kwargs))
print("Dict rewards:   ", spelling_bee_dictionary_reward(test_completions, **test_kwargs))
print("Length rewards:  ", spelling_bee_length_reward(test_completions, **test_kwargs))
print("Pangram rewards:", spelling_bee_pangram_reward(test_completions, **test_kwargs))

Format rewards:  [1.0, 1.0, 0.0, 1.0, 1.0]
Letter rewards:  [1.0, 1.0, 0.0, 1.0, 1.0]
Dict rewards:    [1.0, 1.0, 0.0, 1.0, 1.0]
Length rewards:   [0.5, 0.5, 0.0, 0.1, 0.8]
Pangram rewards: [0.0, 0.0, 0.0, 0.0, 0.0]


## 4. Dataset Builder

In [ ]:
# ============================================================================
# DATASET BUILDERS
# [Improvement 6] Curriculum learning — sort puzzles by difficulty (# valid words)
# [Improvement 1] Include precomputed candidate words per puzzle
# ============================================================================

def load_spelling_bee_dataset(csv_path=None, max_puzzles=None, curriculum=True):
    """Wrap the repo's SpellingBeeDataset into a HuggingFace Dataset with
    the extra columns the reward functions need (letters, center, valid_words).

    [Improvement 6] If curriculum=True, sorts puzzles from easiest (most valid
    words) to hardest (fewest valid words) so the model learns incrementally."""
    sb_ds = SpellingBeeDataset(csv_path=csv_path or SB_CSV_PATH)
    rows = []
    for i, raw_row in enumerate(sb_ds._rows):
        if max_puzzles and i >= max_puzzles:
            break
        item = sb_ds[i]  # {"prompt": [...], "puzzle_id": int}
        letters_str = "".join(sorted(raw_row["letters"]))

        # [Improvement 1] Precompute candidate words for this puzzle
        candidates = get_sb_candidates(raw_row["letters"], raw_row["center"])
        num_solutions = len(raw_row["solutions"])

        rows.append({
            "prompt": item["prompt"],
            "letters": letters_str,
            "center": raw_row["center"],
            "valid_words": ";".join(raw_row["solutions"]),
            "num_solutions": num_solutions,  # for curriculum sorting
            "num_candidates": len(candidates),  # for curriculum sorting
        })

    if curriculum:
        # [Improvement 6] Sort by number of solutions descending (easy first)
        rows.sort(key=lambda r: r["num_solutions"], reverse=True)
        print(f"Curriculum learning enabled: puzzles sorted easy→hard")
        print(f"  Easiest puzzle: {rows[0]['num_solutions']} solutions")
        print(f"  Hardest puzzle: {rows[-1]['num_solutions']} solutions")

    return Dataset.from_list(rows)


def load_wordle_dataset(max_words=500, solutions_path=None, max_guesses=6):
    """Build a history-aware GRPO prompt dataset for Wordle.

    For each target word, creates TWO training prompts:
      1. A blind first guess (no prior history)
      2. A prompt with 1-4 pre-rolled prior guesses and tile feedback
         (🟩🟨⬜), so the model learns to reason about game history.

    [Improvement 5] First 5 blind-guess prompts always use entropy-optimal
    starting words (SOARE, ROATE, SLATE, CRANE, TRACE) as targets to
    emphasize strong openers.

    The model generates ONE completion (the next guess) and is rewarded
    by the same reward functions as before."""
    solutions_path = solutions_path or WORDLE_SOLUTIONS_PATH
    targets = [w.strip().upper() for w in open(solutions_path).readlines() if w.strip()]
    targets = targets[:max_words]

    word_set = load_dictionary(length=5)
    # Pool of decoy words (exclude targets so prior guesses are never correct)
    target_set = set(targets)
    decoy_pool = [w for w in word_set if w not in target_set]

    user_msg = WORDLE_USER_PROMPT_TEMPLATE.format(max_guesses=max_guesses)
    rows = []

    for target in targets:
        # Create multiple training prompts per target: one blind, one with history
        for variant in range(2):
            if variant == 0:
                # Variant 0: blind first guess
                n_prior = 0
            else:
                # Variant 1: 1-4 prior guesses with tile feedback
                n_prior = random.randint(1, 4)

            msgs = [
                {"role": "system", "content": WORDLE_SYSTEM_PROMPT},
                {"role": "user",   "content": user_msg},
            ]

            if n_prior > 0:
                priors = random.sample(decoy_pool, n_prior)
                for k, guess in enumerate(priors):
                    tiles = _score_guess(guess, target)
                    feedback_str = _format_feedback(guess, tiles)
                    msgs.append({"role": "assistant", "content": guess})
                    remaining = max_guesses - (k + 1)
                    msgs.append({"role": "user", "content":
                        f"Feedback: {feedback_str}\n"
                        f"You have {remaining} attempts remaining.\n"
                        f"Please guess another word."})

            rows.append({
                "prompt": msgs,
                "target_word": target,
            })

    random.shuffle(rows)
    return Dataset.from_list(rows)

In [ ]:
if GAME == "spelling_bee":
    dataset = load_spelling_bee_dataset(CSV_PATH, max_puzzles=MAX_PUZZLES)
    reward_funcs = [
        spelling_bee_format_reward,
        spelling_bee_letter_reward,         # [Improvement 2] partial rewards
        real_word_reward,
        spelling_bee_dictionary_reward,
        spelling_bee_length_reward,
        spelling_bee_pangram_reward,         # [Improvement 2] stronger pangram reward
        spelling_bee_frequency_reward,       # [Improvement 3] word frequency prior
        spelling_bee_candidate_rank_reward,  # [Improvement 12] candidate ranking
    ]
else:
    dataset = load_wordle_dataset()
    reward_funcs = [
        wordle_format_reward,
        wordle_letter_match_reward,
        wordle_exact_match_reward,
        real_word_reward,
        wordle_valid_guess_reward,
        wordle_info_gain_reward,             # [Improvement 4] information gain
        wordle_repeat_penalty_reward,        # [Improvement 7] penalize repeats
    ]

print(f"Loaded {len(dataset)} training prompts for '{GAME}'")
print(f"Using {len(reward_funcs)} reward functions")
print(f"\nSample prompt:\n{dataset[0]['prompt'][0]['content'][:200]}...")

Loaded 200 training prompts for 'spelling_bee'
Using 6 reward functions

Sample prompt:
You are an expert NYT Spelling Bee player. In this game you must guess words that:
  1. Are at least 4 letters long.
  2. Use only the allowed letters (letters may be reused).
  3. Always contain the ...


## 5. Load Model & Tokenizer

In [9]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32,
    device_map="auto",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")
print(f"Dtype: {next(model.parameters()).dtype}")
print(f"Device: {next(model.parameters()).device}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Model loaded: Qwen/Qwen2.5-1.5B-Instruct
Parameters: 1543.7M
Dtype: torch.bfloat16
Device: cuda:0


In [10]:
peft_config = None
if USE_LORA:
    peft_config = LoraConfig(
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    print(f"LoRA config: r={LORA_R}, alpha={LORA_ALPHA}")
    trainable = LORA_R * 2 * 7
    print(f"Estimated trainable params: ~{trainable * 28 / 1e3:.0f}K per attention block")
else:
    print("Full fine-tuning (no LoRA)")

LoRA config: r=16, alpha=32
Estimated trainable params: ~6K per attention block


## 5.5 SFT Warmstart

Before GRPO, do a quick supervised fine-tuning pass so the model learns what
valid Spelling Bee answers look like.

In [ ]:
from trl import SFTConfig, SFTTrainer

# ============================================================================
# SFT DATASET BUILDERS
# [Improvement 10] Improved SFT examples with reasoning
# [Improvement 11] "Thinking" format: Reasoning → Answer
# [Improvement 5] Entropy-optimal first guesses for Wordle
# ============================================================================

# --- [Improvement 11] Thinking-format system prompts ---
SB_SYSTEM_PROMPT_THINKING = """You are an expert NYT Spelling Bee player. In this game you must guess words that:
  1. Are at least 4 letters long.
  2. Use only the allowed letters (letters may be reused).
  3. Always contain the center letter.
A pangram uses every allowed letter at least once and earns a bonus.

Think step by step, then give your answer.
Format your response as:
Reasoning:
- (your reasoning about which letters to use)
Answer:
(single word guess)"""

WORDLE_SYSTEM_PROMPT_THINKING = """You are an expert Wordle player. In this game you must guess a secret 5-letter word in 6 tries.
After each guess you receive feedback for every letter:
  - GREEN: the letter is correct and in the right position.
  - YELLOW: the letter is in the word but in the wrong position.
  - GRAY: the letter is not in the word at all.
Use the feedback to narrow down possibilities.

Think step by step, then give your answer.
Format your response as:
Reasoning:
- (your reasoning about the feedback)
Answer:
(single 5-letter word guess)"""

# Use thinking-format prompts for training
USE_THINKING_FORMAT = True


def build_sft_dataset(csv_path=None, max_puzzles=None, solutions_per_puzzle=10):
    """[Improvement 10] Create (prompt, completion) pairs from puzzle solutions.
    [Improvement 11] Includes reasoning/thinking format examples."""
    sb_ds = SpellingBeeDataset(csv_path=csv_path or SB_CSV_PATH)
    rows = []
    system_prompt = SB_SYSTEM_PROMPT_THINKING if USE_THINKING_FORMAT else SB_SYSTEM_PROMPT

    for i, raw_row in enumerate(sb_ds._rows):
        if max_puzzles and i >= max_puzzles:
            break
        item = sb_ds[i]
        solutions = list(raw_row["solutions"])
        allowed = sorted(raw_row["letters"])
        center = raw_row["center"]
        sampled = random.sample(solutions, min(solutions_per_puzzle, len(solutions)))

        for word in sampled:
            # Construct the user message
            user_msg = SB_USER_PROMPT_TEMPLATE.format(
                letters=", ".join(allowed),
                center=center,
                max_guesses=10,
            )

            if USE_THINKING_FORMAT:
                # [Improvement 11] Build reasoning chain
                is_pangram = set(word.upper()) >= set(l.upper() for l in raw_row["letters"])
                reasoning_lines = [
                    f"- The center letter {center} must appear in my word",
                    f"- Allowed letters are: {', '.join(allowed)}",
                    f"- '{word}' contains center letter {center}: {'yes' if center.upper() in word.upper() else 'no'}",
                    f"- '{word}' uses only allowed letters: yes",
                    f"- '{word}' is at least 4 letters long: {'yes' if len(word) >= 4 else 'no'}",
                ]
                if is_pangram:
                    reasoning_lines.append(f"- '{word}' uses ALL allowed letters — it's a pangram!")
                answer = f"Reasoning:\n" + "\n".join(reasoning_lines) + f"\nAnswer:\n{word}"
            else:
                answer = word

            rows.append({
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_msg},
                    {"role": "assistant", "content": answer},
                ]
            })

    random.shuffle(rows)
    return Dataset.from_list(rows)

sft_dataset = build_sft_dataset(CSV_PATH, max_puzzles=MAX_PUZZLES, solutions_per_puzzle=10)
print(f"SFT dataset: {len(sft_dataset)} examples")
print(f"Thinking format: {'ENABLED' if USE_THINKING_FORMAT else 'DISABLED'}")
print(f"Sample:\n  Prompt: {sft_dataset[0]['messages'][1]['content'][:100]}...")
print(f"  Answer: {sft_dataset[0]['messages'][2]['content'][:200]}")

SFT dataset: 1000 examples
Sample:
  Prompt: New Spelling Bee puzzle!
Allowed letters: A, B, C, E, K, L, M
Center letter (must appear in every wo...
  Answer: ABACK


In [ ]:
SFT_OUTPUT_DIR = "./sft_warmstart"
SFT_EPOCHS = 2    # 2 epochs for stronger warmstart
SFT_BATCH_SIZE = 4
SFT_LR = 2e-5

sft_args = SFTConfig(
    output_dir=SFT_OUTPUT_DIR,
    num_train_epochs=SFT_EPOCHS,
    per_device_train_batch_size=SFT_BATCH_SIZE,
    learning_rate=SFT_LR,
    bf16=USE_BF16,
    logging_steps=10,
    save_steps=100,
    save_total_limit=1,
    report_to="none",
)

sft_trainer = SFTTrainer(
    model=model,
    args=sft_args,
    train_dataset=sft_dataset,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(f"SFT warmstart: {len(sft_dataset)} examples, {SFT_EPOCHS} epoch(s)")
print(f"Estimated SFT steps: ~{len(sft_dataset) * SFT_EPOCHS // SFT_BATCH_SIZE}")

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

SFT warmstart: 1000 examples, 1 epoch(s)
Estimated SFT steps: ~250


In [13]:
# Run SFT warmstart — takes ~5-10 minutes on a T4
sft_trainer.train()
sft_trainer.save_model(SFT_OUTPUT_DIR)
tokenizer.save_pretrained(SFT_OUTPUT_DIR)
print(f"SFT warmstart model saved to {SFT_OUTPUT_DIR}")

# The model variable is now the SFT-warmed model (LoRA adapter applied in-place).
# GRPO training below will continue fine-tuning from this checkpoint.
# If using LoRA, merge SFT adapter so GRPO starts fresh with a new adapter.
if USE_LORA:
    model = sft_trainer.model.merge_and_unload()
    print("SFT LoRA merged into base model. GRPO will train a new LoRA adapter on top.")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
10,3.170538
20,2.377165
30,1.794385
40,1.213871
50,0.636987
60,0.314119
70,0.195426
80,0.176992
90,0.168330
100,0.166074


SFT warmstart model saved to ./sft_warmstart
SFT LoRA merged into base model. GRPO will train a new LoRA adapter on top.


## 6. GRPO Training

In [ ]:
# [Improvement 11] Increase max_completion_length when using thinking format
_max_comp_len = 128 if USE_THINKING_FORMAT else MAX_COMPLETION_LENGTH

training_args = GRPOConfig(
    output_dir=OUTPUT_DIR,

    num_generations=NUM_GENERATIONS,
    max_completion_length=_max_comp_len,
    temperature=TEMPERATURE,
    beta=BETA,

    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    bf16=USE_BF16,
    logging_steps=1,
    save_steps=50,
    save_total_limit=3,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    report_to="none",
    log_level="info",
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=reward_funcs,
    train_dataset=dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)

print(f"Trainer created. Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
print(f"Max completion length: {_max_comp_len} ({'thinking format' if USE_THINKING_FORMAT else 'standard'})")
print(f"Total training steps: ~{len(dataset) * NUM_EPOCHS // (BATCH_SIZE * GRAD_ACCUM_STEPS)}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Trainer created. Effective batch size: 8
Total training steps: ~75


In [15]:
trainer.train()

***** Running training *****
  Num examples = 200
  Num Epochs = 3
  Instantaneous batch size per device = 2
  Total train batch size (w. parallel, distributed & accumulation) = 8
  Gradient Accumulation steps = 4
  Total optimization steps = 600
  Number of trainable parameters = 18,464,768
Passing `generation_config` together with generation-related arguments=({'disable_compile'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


Step,Training Loss
1,0.539834
2,0.225590
3,0.359230
4,-0.018990
5,0.102675
6,0.036754
7,0.726026
8,0.098987
9,0.128193
10,0.103506


Saving model checkpoint to ./grpo_output/checkpoint-50
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",

TrainOutput(global_step=600, training_loss=0.11003860711236485, metrics={'train_runtime': 3676.6434, 'train_samples_per_second': 0.163, 'train_steps_per_second': 0.163, 'total_flos': 0.0, 'train_loss': 0.11003860711236485})

In [16]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

Saving model checkpoint to ./grpo_output
loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_att

Model saved to ./grpo_output


## 7. Inference & Testing with Spelling Bee Environment

In [17]:
print("SpellingBeeConfig:", SpellingBeeConfig)
print("SpellingBeeEnv:", SpellingBeeEnv)
print("WordleConfig:", WordleConfig)
print("WordleEnv:", WordleEnv)

SpellingBeeConfig: <class 'nytgames.env.spellingbee.SpellingBeeConfig'>
SpellingBeeEnv: <class 'nytgames.env.spellingbee.SpellingBeeEnv'>
WordleConfig: <class 'nytgames.env.wordle.WordleConfig'>
WordleEnv: <class 'nytgames.env.wordle.WordleEnv'>


In [ ]:
# ============================================================================
# INFERENCE UTILITIES
# [Improvement 8] Beam search during evaluation
# [Improvement 1] Constrained generation from valid candidates
# ============================================================================

def load_trained_model(model_path):
    model_path = Path(model_path)
    adapter_config_path = model_path / "adapter_config.json"

    if adapter_config_path.exists():
        with open(adapter_config_path) as f:
            adapter_cfg = json.load(f)
        base_model_name = adapter_cfg.get("base_model_name_or_path", adapter_cfg.get("base_model"))
        print(f"Loading base model: {base_model_name}")
        base_model = AutoModelForCausalLM.from_pretrained(
            base_model_name,
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        inf_model = PeftModel.from_pretrained(base_model, str(model_path))
        inf_tokenizer = AutoTokenizer.from_pretrained(str(model_path))
    else:
        inf_model = AutoModelForCausalLM.from_pretrained(
            str(model_path),
            torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
            device_map="auto",
        )
        inf_tokenizer = AutoTokenizer.from_pretrained(str(model_path))

    if inf_tokenizer.pad_token is None:
        inf_tokenizer.pad_token = inf_tokenizer.eos_token
    inf_model.eval()
    return inf_model, inf_tokenizer


def generate_guess(inf_model, inf_tokenizer, messages, temperature=0.7,
                   use_beam_search=False, num_beams=10):
    """Generate a guess from the model.

    [Improvement 8] If use_beam_search=True, uses beam search (deterministic)
    instead of sampling for more reliable inference."""
    text = inf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = inf_tokenizer(text, return_tensors="pt").to(inf_model.device)

    with torch.no_grad():
        if use_beam_search:
            # [Improvement 8] Beam search for evaluation
            outputs = inf_model.generate(
                **inputs,
                max_new_tokens=64 if USE_THINKING_FORMAT else 32,
                num_beams=num_beams,
                do_sample=False,
                early_stopping=True,
            )
        else:
            outputs = inf_model.generate(
                **inputs,
                max_new_tokens=64 if USE_THINKING_FORMAT else 32,
                temperature=temperature,
                do_sample=True,
                top_p=0.95,
            )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    raw_output = inf_tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # [Improvement 11] Extract answer from thinking format
    return _extract_answer(raw_output)


def generate_from_candidates(inf_model, inf_tokenizer, messages, candidates,
                             temperature=0.7, top_k=20):
    """[Improvement 1] Generate by scoring candidate words and picking the best.
    Instead of free-form generation, rank candidates by model likelihood.

    [Improvement 12] This implements candidate ranking: the model acts as a
    policy over dictionary words."""
    if not candidates:
        return generate_guess(inf_model, inf_tokenizer, messages, temperature)

    text = inf_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = inf_tokenizer(text, return_tensors="pt").to(inf_model.device)

    # Score a random subset for efficiency
    if len(candidates) > top_k:
        sample = random.sample(candidates, top_k)
    else:
        sample = list(candidates)

    scores = []
    with torch.no_grad():
        for word in sample:
            word_ids = inf_tokenizer.encode(word, add_special_tokens=False, return_tensors="pt").to(inf_model.device)
            combined = torch.cat([inputs["input_ids"], word_ids], dim=1)
            outputs = inf_model(combined)
            # Score = sum of log probs for the word tokens
            logits = outputs.logits[0, inputs["input_ids"].shape[1]-1:-1, :]
            log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
            word_log_prob = sum(
                log_probs[t, word_ids[0, t]].item()
                for t in range(word_ids.shape[1])
            )
            scores.append(word_log_prob)

    # Sample from top candidates (temperature-weighted)
    scores_t = torch.tensor(scores)
    if temperature > 0:
        probs = torch.nn.functional.softmax(scores_t / temperature, dim=0)
        idx = torch.multinomial(probs, 1).item()
    else:
        idx = scores_t.argmax().item()

    return sample[idx]


# --- [Improvement 8] Default to beam search during evaluation ---
USE_BEAM_SEARCH_EVAL = True
EVAL_NUM_BEAMS = 10

inf_model, inf_tokenizer = load_trained_model(OUTPUT_DIR)

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_

Loading base model: Qwen/Qwen2.5-1.5B-Instruct


loading weights file model.safetensors from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/model.safetensors
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "eos_token_id": 151645,
  "output_attentions": false,
  "output_hidden_states": false,
  "use_cache": true
}

We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

Model config PreTrainedConfig {
  "transformers_version": "5.2.0"
}



### 7.1 Run a Spelling Bee Game

In [ ]:
# ============================================================================
# SPELLING BEE INFERENCE
# [Improvement 1] Constrained generation from precomputed candidates
# [Improvement 8] Beam search during evaluation
# ============================================================================

config = SpellingBeeConfig(
    center_letter="H",
    letter_set={"G", "I", "P", "R", "A", "C", "H"},
    word_set={
        "GRAPHIC", "HIGHCHAIR", "PARAGRAPH", "ARCHAIC", "CHICHI", "PARIAH",
        "AARGH", "CHAIR", "CHICA", "CHIRP", "GRAPH", "PARCH",
        "ARCH", "CHAI", "CHAP", "CHAR", "CHIA", "CHIC", "CHIP",
        "HAIR", "HARP", "HIGH", "RICH",
    },
    max_guesses=10,
)

env = SpellingBeeEnv(config)
obs, info = env.reset()

prompt_text = SB_USER_PROMPT_TEMPLATE.format(
    letters=", ".join(sorted(config.letter_set)),
    center=config.center_letter,
    max_guesses=config.max_guesses,
)

# Use thinking-format system prompt if enabled
system_prompt = SB_SYSTEM_PROMPT_THINKING if USE_THINKING_FORMAT else SB_SYSTEM_PROMPT
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": prompt_text},
]

print(f"{'='*60}")
print(f"Spelling Bee — Letters: {sorted(config.letter_set)}, Center: {config.center_letter}")
print(f"Valid words: {len(config.word_set)}, Max guesses: {config.max_guesses}")
print(f"Beam search: {USE_BEAM_SEARCH_EVAL}, Thinking format: {USE_THINKING_FORMAT}")
print(f"{'='*60}\n")

# [Improvement 1] Precompute candidate words for constrained generation
sb_candidates = get_sb_candidates(config.letter_set, config.center_letter)
print(f"Precomputed {len(sb_candidates)} dictionary candidates for this puzzle\n")

already_guessed = set()
allowed_letters = set(l.upper() for l in config.letter_set)
center = config.center_letter.upper()
MAX_RETRIES = 10

def is_valid_sb_guess(w):
    """Check if a guess uses only allowed letters, contains center, and is >=4 chars."""
    return (len(w) >= 4
            and set(w) <= allowed_letters
            and center in w
            and w not in already_guessed)

for turn in range(config.max_guesses):
    # [Improvement 1] Try constrained generation first, fall back to free generation
    remaining_candidates = [c for c in sb_candidates if c not in already_guessed]

    if remaining_candidates:
        # Use candidate ranking to pick from valid dictionary words
        word = generate_from_candidates(
            inf_model, inf_tokenizer, messages,
            remaining_candidates, temperature=0.7, top_k=30
        ).upper()
    else:
        # Fall back to free generation with retries
        word = None
        best_fallback = None
        for attempt in range(MAX_RETRIES):
            raw_guess = generate_guess(
                inf_model, inf_tokenizer, messages,
                temperature=1.0,
                use_beam_search=USE_BEAM_SEARCH_EVAL,
                num_beams=EVAL_NUM_BEAMS,
            )
            candidate = raw_guess.split()[0] if raw_guess.split() else raw_guess
            candidate = "".join(c for c in candidate if c.isalpha()).upper()
            if is_valid_sb_guess(candidate):
                word = candidate
                break
            if candidate not in already_guessed and best_fallback is None:
                best_fallback = candidate
        if word is None:
            word = best_fallback or candidate

    already_guessed.add(word)
    messages.append({"role": "assistant", "content": word})

    obs, reward, terminated, truncated, info = env.step(word)
    print(f"Turn {turn+1:2d} | Guess: {word:15s} | Reward: {reward:2.0f} | Total: {obs['total_points']:3d}")

    if terminated or truncated:
        break

    if reward > 0:
        fb = f"'{word}' was correct! You earned {reward:.0f} point(s). Running total: {obs['total_points']}."
    else:
        fb = f"'{word}' was not accepted. {obs['feedback']}"
    guessed = ", ".join(obs["valid_words_guessed"]) if obs["valid_words_guessed"] else "none"
    fb += (f"\nGuesses used: {obs['num_guesses']}. Words so far: {guessed}."
           f"\nREMINDER: You may ONLY use the letters {', '.join(sorted(allowed_letters))} "
           f"(center letter {center} must appear). Words must be at least 4 letters."
           f"\nPlease guess another word. Do NOT repeat any previous guess.")
    messages.append({"role": "user", "content": fb})

print(f"\n{'='*60}")
print(f"Game Over! Final score: {obs['total_points']} in {obs['num_guesses']} guesses")
found_words = sorted(w for w, r, _ in info['history'] if r > 0)
print(f"Words found: {found_words}")
print(f"{'='*60}")

Spelling Bee — Letters: ['A', 'C', 'G', 'H', 'I', 'P', 'R'], Center: H
Valid words: 23, Max guesses: 10

Turn  1 | Guess: HARP            | Reward:  1 | Total:   1
Turn  2 | Guess: GRAPES          | Reward:  0 | Total:   1
Turn  3 | Guess: HARPSE          | Reward:  0 | Total:   1
Turn  4 | Guess: HARPS           | Reward:  0 | Total:   1
Turn  5 | Guess: HARPSE          | Reward:  0 | Total:   1
Turn  6 | Guess: PARCHES         | Reward:  0 | Total:   1
Turn  7 | Guess: GRAPEHAWK       | Reward:  0 | Total:   1
Turn  8 | Guess: RAPHIA          | Reward:  0 | Total:   1
Turn  9 | Guess: HARPER          | Reward:  0 | Total:   1
Turn 10 | Guess: HARRIER         | Reward:  0 | Total:   1

Game Over! Final score: 1 in 10 guesses
Words found: ['HARP']


### 7.2 Batch Evaluation

In [ ]:
def evaluate_on_puzzles(inf_model, inf_tokenizer, csv_path=None, num_puzzles=10, max_guesses=10):
    """Evaluate on Spelling Bee puzzles with all improvements:
    [Improvement 1] Constrained candidate generation
    [Improvement 8] Beam search"""
    sb_ds = SpellingBeeDataset(csv_path=csv_path or SB_CSV_PATH, max_guesses=max_guesses)
    results = []
    system_prompt = SB_SYSTEM_PROMPT_THINKING if USE_THINKING_FORMAT else SB_SYSTEM_PROMPT

    for idx in range(min(num_puzzles, len(sb_ds))):
        item = sb_ds[idx]
        config = sb_ds.get_config(item["puzzle_id"], max_guesses=max_guesses)

        env = SpellingBeeEnv(config)
        obs, info = env.reset()

        # Replace system prompt with thinking-format version
        messages = list(item["prompt"])
        if USE_THINKING_FORMAT:
            messages[0] = {"role": "system", "content": system_prompt}

        # [Improvement 1] Precompute candidates
        sb_candidates = get_sb_candidates(config.letter_set, config.center_letter)

        already_guessed = set()
        allowed_letters = set(l.upper() for l in config.letter_set)
        center = config.center_letter.upper()

        def is_valid_sb(w, _al=allowed_letters, _c=center, _ag=already_guessed):
            return (len(w) >= 4
                    and set(w) <= _al
                    and _c in w
                    and w not in _ag)

        for _ in range(max_guesses):
            remaining_candidates = [c for c in sb_candidates if c not in already_guessed]

            if remaining_candidates:
                # [Improvement 1] Constrained generation
                word = generate_from_candidates(
                    inf_model, inf_tokenizer, messages,
                    remaining_candidates, temperature=0.7, top_k=30
                ).upper()
            else:
                word = None
                best_fallback = None
                for attempt in range(10):
                    raw = generate_guess(
                        inf_model, inf_tokenizer, messages,
                        temperature=1.0,
                        use_beam_search=USE_BEAM_SEARCH_EVAL,
                        num_beams=EVAL_NUM_BEAMS,
                    )
                    candidate = raw.split()[0] if raw.split() else raw
                    candidate = "".join(c for c in candidate if c.isalpha()).upper()
                    if is_valid_sb(candidate):
                        word = candidate
                        break
                    if candidate not in already_guessed and best_fallback is None:
                        best_fallback = candidate
                if word is None:
                    word = best_fallback or candidate

            already_guessed.add(word)
            messages.append({"role": "assistant", "content": word})

            obs, reward, terminated, truncated, info = env.step(word)

            if reward > 0:
                fb = f"'{word}' was correct! Running total: {obs['total_points']}."
            else:
                fb = f"'{word}' was not accepted. {obs['feedback']}"
            fb += (f"\nREMINDER: Only use letters {', '.join(sorted(allowed_letters))} "
                   f"(center {center} required, min 4 letters)."
                   f"\nPlease guess another word. Do NOT repeat any previous guess.")
            messages.append({"role": "user", "content": fb})

            if terminated or truncated:
                break

        letters_str = "".join(sorted(config.letter_set))
        results.append({
            "puzzle_id": item["puzzle_id"],
            "letters": letters_str,
            "center": config.center_letter,
            "available_words": len(config.word_set),
            "found": len(obs["valid_words_guessed"]),
            "score": obs["total_points"],
        })
        print(f"Puzzle {item['puzzle_id']:3d} | {letters_str} (center={config.center_letter}) | "
              f"Found {len(obs['valid_words_guessed'])}/{len(config.word_set)} words | Score: {obs['total_points']}")

    avg_found = sum(r["found"] for r in results) / len(results)
    avg_score = sum(r["score"] for r in results) / len(results)
    print(f"\n{'='*60}")
    print(f"Evaluated {len(results)} puzzles")
    print(f"Avg words found: {avg_found:.1f}")
    print(f"Avg score: {avg_score:.1f}")
    print(f"{'='*60}")
    return results


eval_results = evaluate_on_puzzles(inf_model, inf_tokenizer, CSV_PATH, num_puzzles=10, max_guesses=10)

Puzzle   1 | AHORTWY (center=W) | Found 6/26 words | Score: 23
Puzzle   2 | CFIMNOR (center=I) | Found 0/36 words | Score: 0
Puzzle   3 | CEFHILY (center=C) | Found 1/28 words | Score: 6
Puzzle   4 | ABDGIRT (center=T) | Found 1/30 words | Score: 5
Puzzle   5 | EGHILTW (center=I) | Found 1/45 words | Score: 1
Puzzle   6 | FLMNORU (center=R) | Found 0/29 words | Score: 0
Puzzle   7 | ACEHKNY (center=A) | Found 2/31 words | Score: 10
Puzzle   8 | BEHILMT (center=T) | Found 0/41 words | Score: 0
Puzzle   9 | CFILMOR (center=O) | Found 0/38 words | Score: 0
Puzzle  10 | AGHOPRT (center=A) | Found 1/60 words | Score: 1

Evaluated 10 puzzles
Avg words found: 1.2
Avg score: 4.6


In [ ]:
## 8. Wordle GRPO Training

# Train a separate model for Wordle using the same pipeline (SFT warmstart → GRPO).

In [ ]:
WORDLE_OUTPUT_DIR = "./grpo_output_wordle"
WORDLE_MAX_PUZZLES = 200

# Reload fresh base model (Spelling Bee training modified the model in-place)
wordle_base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16 if USE_BF16 else torch.float32,
    device_map="auto",
)
wordle_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if wordle_tokenizer.pad_token is None:
    wordle_tokenizer.pad_token = wordle_tokenizer.eos_token

# Build Wordle dataset and reward functions (with new improvements)
wordle_dataset = load_wordle_dataset(max_words=WORDLE_MAX_PUZZLES)
wordle_reward_funcs = [
    wordle_format_reward,
    wordle_letter_match_reward,          # [Improvement 2] partial rewards
    wordle_exact_match_reward,
    real_word_reward,
    wordle_valid_guess_reward,
    wordle_info_gain_reward,             # [Improvement 4] information gain
    wordle_repeat_penalty_reward,        # [Improvement 7] penalize repeats
]
print(f"Wordle dataset: {len(wordle_dataset)} prompts, {len(wordle_reward_funcs)} reward functions")

In [ ]:
# ============================================================================
# WORDLE SFT WARMSTART
# [Improvement 5] Entropy-optimal first guesses (SOARE, ROATE, SLATE, CRANE, TRACE)
# [Improvement 10] Improved multi-turn SFT with feedback reasoning
# [Improvement 11] Thinking format for Wordle
# ============================================================================

# [Improvement 5] Best entropy-optimal starting words
OPTIMAL_FIRST_GUESSES = ["SOARE", "ROATE", "SLATE", "CRANE", "TRACE"]

def build_wordle_sft_dataset(solutions_path=None, max_words=200):
    """Build multi-turn SFT pairs with reasoning and optimal starts.
    Includes:
    - Direct guess examples (teaches word generation)
    - Multi-turn games with feedback (teaches feedback reasoning)
    - [Improvement 5] Optimal first guess examples
    - [Improvement 11] Thinking/reasoning format"""
    solutions_path = solutions_path or WORDLE_SOLUTIONS_PATH
    words = [w.strip().upper() for w in open(solutions_path).readlines() if w.strip()]
    words = words[:max_words]
    word_set = load_dictionary(length=5)
    word_list = [w for w in word_set if w not in set(words)]
    user_msg = WORDLE_USER_PROMPT_TEMPLATE.format(max_guesses=6)
    system_prompt = WORDLE_SYSTEM_PROMPT_THINKING if USE_THINKING_FORMAT else WORDLE_SYSTEM_PROMPT

    rows = []

    # --- [Improvement 5] Add optimal first guess examples ---
    for starter in OPTIMAL_FIRST_GUESSES:
        if starter in word_set:
            if USE_THINKING_FORMAT:
                answer = (f"Reasoning:\n"
                          f"- This is my first guess with no information yet\n"
                          f"- '{starter}' is a strong opener because it tests common letters\n"
                          f"- It covers frequently used vowels and consonants\n"
                          f"Answer:\n{starter}")
            else:
                answer = starter

            # Create multiple examples with different targets but same first guess
            for target in random.sample(words, min(5, len(words))):
                rows.append({"messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_msg},
                    {"role": "assistant", "content": answer},
                ]})

    for target in words:
        # Version 1: direct correct guess
        if USE_THINKING_FORMAT:
            answer = (f"Reasoning:\n"
                      f"- I need to guess a 5-letter word\n"
                      f"- '{target}' is a valid English word\n"
                      f"Answer:\n{target}")
        else:
            answer = target

        rows.append({"messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": answer},
        ]})

        # Version 2: 1-3 prior guesses with tile feedback, then correct answer
        n_prior = random.randint(1, 3)
        priors = random.sample(word_list, n_prior)
        msgs = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg},
        ]

        for k, guess in enumerate(priors):
            tiles = _score_guess(guess, target)
            feedback_str = _format_feedback(guess, tiles)

            if USE_THINKING_FORMAT:
                guess_answer = (f"Reasoning:\n"
                                f"- {'This is my first guess, testing common letters' if k == 0 else 'Based on previous feedback, narrowing down candidates'}\n"
                                f"Answer:\n{guess}")
            else:
                guess_answer = guess

            msgs.append({"role": "assistant", "content": guess_answer})
            remaining = 6 - (k + 1)
            msgs.append({"role": "user", "content":
                f"Feedback: {feedback_str}\n"
                f"You have {remaining} attempts remaining.\n"
                f"Please guess another word."})

        # [Improvement 10] Final correct answer with reasoning about feedback
        if USE_THINKING_FORMAT:
            # Build reasoning from accumulated tile feedback
            reasoning_lines = ["- Analyzing all feedback received:"]
            for k, guess in enumerate(priors):
                tiles = _score_guess(guess, target)
                for pos, (letter, tile) in enumerate(zip(guess, tiles)):
                    if tile == "correct":
                        reasoning_lines.append(f"  - {letter} is in position {pos+1} (GREEN)")
                    elif tile == "present":
                        reasoning_lines.append(f"  - {letter} is in the word but not position {pos+1} (YELLOW)")
            reasoning_lines.append(f"- '{target}' matches all constraints")

            final_answer = "Reasoning:\n" + "\n".join(reasoning_lines) + f"\nAnswer:\n{target}"
        else:
            final_answer = target

        msgs.append({"role": "assistant", "content": final_answer})
        rows.append({"messages": msgs})

    random.shuffle(rows)
    return Dataset.from_list(rows)

wordle_sft_dataset = build_wordle_sft_dataset(max_words=WORDLE_MAX_PUZZLES)
print(f"Wordle SFT dataset: {len(wordle_sft_dataset)} examples (incl. multi-turn + optimal starts)")
print(f"Thinking format: {'ENABLED' if USE_THINKING_FORMAT else 'DISABLED'}")
print(f"Optimal first guesses added: {OPTIMAL_FIRST_GUESSES}")

wordle_peft_config = LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type="CAUSAL_LM",
)

wordle_sft_args = SFTConfig(
    output_dir="./sft_warmstart_wordle",
    num_train_epochs=2,   # 2 epochs for stronger warmstart
    per_device_train_batch_size=SFT_BATCH_SIZE,
    learning_rate=SFT_LR,
    bf16=USE_BF16,
    logging_steps=10,
    save_steps=100,
    save_total_limit=1,
    report_to="none",
)

wordle_sft_trainer = SFTTrainer(
    model=wordle_base_model,
    args=wordle_sft_args,
    train_dataset=wordle_sft_dataset,
    processing_class=wordle_tokenizer,
    peft_config=wordle_peft_config,
)

print(f"Wordle SFT warmstart: {len(wordle_sft_dataset)} examples, 2 epoch(s)")
wordle_sft_trainer.train()

if USE_LORA:
    wordle_base_model = wordle_sft_trainer.model.merge_and_unload()
    print("Wordle SFT LoRA merged into base model.")

In [ ]:
# --- Wordle GRPO Training ---
wordle_grpo_args = GRPOConfig(
    output_dir=WORDLE_OUTPUT_DIR,
    num_generations=NUM_GENERATIONS,
    max_completion_length=64 if USE_THINKING_FORMAT else 8,  # longer for thinking format
    temperature=1.4,           # higher to prevent mode collapse
    beta=0.01,                 # lower KL penalty — less collapse risk
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    bf16=USE_BF16,
    logging_steps=1,
    save_steps=50,
    save_total_limit=3,
    max_grad_norm=0.3,
    warmup_ratio=0.03,
    report_to="none",
    log_level="info",
)

wordle_trainer = GRPOTrainer(
    model=wordle_base_model,
    args=wordle_grpo_args,
    reward_funcs=wordle_reward_funcs,
    train_dataset=wordle_dataset,
    peft_config=wordle_peft_config,
    processing_class=wordle_tokenizer,
)

print(f"Wordle trainer created. Training steps: ~{len(wordle_dataset) * NUM_EPOCHS // (BATCH_SIZE * GRAD_ACCUM_STEPS)}")
wordle_trainer.train()

wordle_trainer.save_model(WORDLE_OUTPUT_DIR)
wordle_tokenizer.save_pretrained(WORDLE_OUTPUT_DIR)
print(f"Wordle model saved to {WORDLE_OUTPUT_DIR}")

## 9. Inference & Testing with Wordle Environment

In [ ]:
# Load the Wordle-trained model for evaluation
inf_model, inf_tokenizer = load_trained_model(WORDLE_OUTPUT_DIR)

### 9.1 Run a Single Wordle Game

In [ ]:
# ============================================================================  
# SINGLE WORDLE GAME
# [Improvement 5] Start with entropy-optimal first guess
# [Improvement 7] Penalize repeated guesses
# [Improvement 8] Beam search during evaluation
# ============================================================================  

WORDLE_WORD_SET = load_dictionary(length=5)

target_word = "CRANE"

wordle_cfg = WordleConfig(
    target_word=target_word,
    word_set=WORDLE_WORD_SET,
    max_guesses=6,
)

env = WordleEnv(wordle_cfg)
obs, info = env.reset()

user_msg = WORDLE_USER_PROMPT_TEMPLATE.format(max_guesses=wordle_cfg.max_guesses)
system_prompt = WORDLE_SYSTEM_PROMPT_THINKING if USE_THINKING_FORMAT else WORDLE_SYSTEM_PROMPT
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user",   "content": user_msg},
]

print(f"{'='*60}")
print(f"Wordle — Target: {target_word} (hidden from model)")
print(f"Max guesses: {wordle_cfg.max_guesses}")
print(f"Beam search: {USE_BEAM_SEARCH_EVAL}, Thinking format: {USE_THINKING_FORMAT}")
print(f"{'='*60}\n")

# Track known constraints from tile feedback
green_letters = {}    # position -> letter (confirmed correct)
yellow_letters = {}   # position -> set of letters (present but not here)
gray_letters = set()  # letters not in the word at all
previous_guesses = set()  # [Improvement 7] Track previous guesses

def build_constraint_summary():
    """Build a human-readable summary of what we know from tile feedback."""
    lines = []
    if green_letters:
        parts = [f"{letter} is in position {pos+1}" for pos, letter in sorted(green_letters.items())]
        lines.append("Confirmed positions: " + ", ".join(parts) + ".")
    if yellow_letters:
        present = set()
        for pos, letters in yellow_letters.items():
            present.update(letters)
        present -= set(green_letters.values())
        if present:
            lines.append(f"Letters in the word but position unknown: {', '.join(sorted(present))}.")
    if gray_letters:
        truly_gray = gray_letters - set(green_letters.values())
        for letters in yellow_letters.values():
            truly_gray -= letters
        if truly_gray:
            lines.append(f"Letters NOT in the word: {', '.join(sorted(truly_gray))}.")
    return "\n".join(lines)

for turn in range(wordle_cfg.max_guesses):
    # Retry loop: only submit valid 5-letter dictionary words
    word = None
    best_fallback = None

    for attempt in range(20):  # increased retries from 10 to 20
        raw_guess = generate_guess(
            inf_model, inf_tokenizer, messages,
            temperature=0.7 + 0.05 * attempt,  # increase temperature on retries
            use_beam_search=USE_BEAM_SEARCH_EVAL,
            num_beams=EVAL_NUM_BEAMS,
        )
        candidate = raw_guess.split()[0] if raw_guess.split() else raw_guess
        candidate = "".join(c for c in candidate if c.isalpha()).upper()[:5]

        # [Improvement 7] Skip repeated guesses
        if len(candidate) == 5 and candidate in WORDLE_WORD_SET and candidate not in previous_guesses:
            word = candidate
            break
        # Track best non-repeat fallback (valid word, not a repeat)
        if best_fallback is None and len(candidate) == 5 and candidate in WORDLE_WORD_SET and candidate not in previous_guesses:
            best_fallback = candidate

    if word is None:
        if best_fallback is not None:
            word = best_fallback
        else:
            # Last resort: pick a random valid word that hasn't been guessed
            remaining = WORDLE_WORD_SET - previous_guesses
            if remaining:
                word = random.choice(list(remaining))
            else:
                word = candidate  # truly stuck (shouldn't happen with 12k+ words)

    previous_guesses.add(word)  # [Improvement 7]
    messages.append({"role": "assistant", "content": word})

    obs, reward, terminated, truncated, info = env.step(word)

    # Pretty-print the turn
    fb_line = obs["feedback"] if obs["feedback"] else "(invalid)"
    print(f"Turn {turn+1} | Guess: {word:5s} | {fb_line} | Reward: {reward:.0f}")

    if terminated:
        print(f"\n✅ Solved in {obs['num_guesses']} guess(es)!  Total points: {obs['total_points']}")
        break
    if truncated:
        print(f"\n❌ Failed to guess '{target_word}' in {wordle_cfg.max_guesses} tries.  Total points: {obs['total_points']}")
        break

    # Update constraints from tile feedback
    if obs["feedback"] and "Invalid" not in obs["feedback"]:
        tiles = _score_guess(word, target_word)
        for pos, (letter, tile) in enumerate(zip(word, tiles)):
            if tile == "correct":
                green_letters[pos] = letter
            elif tile == "present":
                yellow_letters.setdefault(pos, set()).add(letter)
            else:  # ABSENT
                gray_letters.add(letter)

    constraint_info = build_constraint_summary()
    fb_msg = f"Feedback: {obs['feedback']}\nYou have {wordle_cfg.max_guesses - obs['num_guesses']} attempts remaining."
    if constraint_info:
        fb_msg += f"\n\nWhat we know so far:\n{constraint_info}"
    # [Improvement 7] Explicitly tell the model not to repeat
    fb_msg += f"\nPrevious guesses: {', '.join(sorted(previous_guesses))}. Do NOT repeat any of these."
    fb_msg += "\nPlease guess another 5-letter word. Use what you know from the feedback."
    messages.append({"role": "user", "content": fb_msg})

env.render()

### 9.2 Batch Wordle Evaluation

In [ ]:
def evaluate_wordle(inf_model, inf_tokenizer, num_puzzles=20, max_guesses=6,
                    solutions_path=None):
    """Play full Wordle games with all improvements:
    [Improvement 7] Penalize repeated guesses
    [Improvement 8] Beam search during evaluation"""
    solutions_path = solutions_path or WORDLE_SOLUTIONS_PATH
    words = [w.strip().upper() for w in open(solutions_path).readlines() if w.strip()]
    words = words[:num_puzzles]

    word_set = load_dictionary(length=5)
    system_prompt = WORDLE_SYSTEM_PROMPT_THINKING if USE_THINKING_FORMAT else WORDLE_SYSTEM_PROMPT
    results = []

    for target in words:
        cfg = WordleConfig(target_word=target, word_set=word_set, max_guesses=max_guesses)
        env = WordleEnv(cfg)
        obs, info = env.reset()

        user_msg = WORDLE_USER_PROMPT_TEMPLATE.format(max_guesses=max_guesses)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_msg},
        ]

        green = {}
        yellow = {}
        gray = set()
        previous_guesses = set()  # [Improvement 7]

        for _ in range(max_guesses):
            # Retry loop: only submit valid dictionary words, no repeats
            word = None
            best_fallback = None
            for attempt in range(10):
                raw = generate_guess(
                    inf_model, inf_tokenizer, messages,
                    temperature=0.7,
                    use_beam_search=USE_BEAM_SEARCH_EVAL,
                    num_beams=EVAL_NUM_BEAMS,
                )
                candidate = raw.split()[0] if raw.split() else raw
                candidate = "".join(c for c in candidate if c.isalpha()).upper()[:5]
                # [Improvement 7] Reject repeated guesses
                if len(candidate) == 5 and candidate in word_set and candidate not in previous_guesses:
                    word = candidate
                    break
                if best_fallback is None and candidate not in previous_guesses:
                    best_fallback = candidate
            if word is None:
                word = best_fallback or candidate

            previous_guesses.add(word)
            messages.append({"role": "assistant", "content": word})
            obs, reward, terminated, truncated, info = env.step(word)

            if terminated or truncated:
                break

            # Update constraints
            if obs["feedback"] and "Invalid" not in obs["feedback"]:
                tiles = _score_guess(word, target)
                for pos, (letter, tile) in enumerate(zip(word, tiles)):
                    if tile == "correct":
                        green[pos] = letter
                    elif tile == "present":
                        yellow.setdefault(pos, set()).add(letter)
                    else:
                        gray.add(letter)

            # Build constraint summary
            hints = []
            if green:
                hints.append("Confirmed: " + ", ".join(f"{l} at position {p+1}" for p, l in sorted(green.items())))
            present = set()
            for s in yellow.values():
                present.update(s)
            present -= set(green.values())
            if present:
                hints.append(f"In word but position unknown: {', '.join(sorted(present))}")
            truly_gray = gray - set(green.values()) - present
            if truly_gray:
                hints.append(f"Not in word: {', '.join(sorted(truly_gray))}")

            fb_msg = f"Feedback: {obs['feedback']}\nYou have {max_guesses - obs['num_guesses']} attempts remaining."
            if hints:
                fb_msg += "\n\nWhat we know:\n" + "\n".join(hints)
            # [Improvement 7] Explicitly tell model not to repeat
            fb_msg += f"\nPrevious guesses: {', '.join(sorted(previous_guesses))}. Do NOT repeat any of these."
            fb_msg += "\nPlease guess another 5-letter word."
            messages.append({"role": "user", "content": fb_msg})

        solved = obs["solved"]
        guesses_used = obs["num_guesses"]
        results.append({
            "target": target,
            "solved": solved,
            "guesses_used": guesses_used,
            "score": obs["total_points"],
        })
        status = f"✅ {guesses_used}/6" if solved else "❌ FAIL"
        print(f"{target}  {status}  | Score: {obs['total_points']}")

    num_solved = sum(r["solved"] for r in results)
    avg_guesses = (sum(r["guesses_used"] for r in results if r["solved"]) / max(num_solved, 1))
    avg_score = sum(r["score"] for r in results) / len(results)

    print(f"\n{'='*60}")
    print(f"Evaluated {len(results)} Wordle puzzles")
    print(f"Solved: {num_solved}/{len(results)} ({100*num_solved/len(results):.1f}%)")
    print(f"Avg guesses (solved only): {avg_guesses:.2f}")
    print(f"Avg score: {avg_score:.1f}")
    print(f"{'='*60}")
    return results


wordle_results = evaluate_wordle(inf_model, inf_tokenizer, num_puzzles=20, max_guesses=6)